# Analysis transcription
## Setup human genes and regions for GO and AME
### Author: Martin Loza
### Date: 26/01/19

In [21]:
# Change R language to English
Sys.setenv(LANGUAGE = "en")

# Init
suppressPackageStartupMessages({
    library(dplyr)
    library(stringr)
    library(ggplot2)
    library(patchwork)
    library(dplyr)
})

# Local variables 
seed = 777
date = "260109"

# Define colors for strand plots
red = "#E41A1C"
blue = "#377EB8"
# Define colors for gene types
green = "#4DAF4A"
purple = "#984EA3"
text_size = 18
width = 18.6
dot_size = 4
line_size = 1.5
dpi = 300

in_dir = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/Data/Annotated_ncRNA_PCG_pairs/"
ensembl_raw_annotation_dir = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/Data/ENSEMBL/selected/"
out_dir = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/04_Figure_window_size_and_GO/Analysis_transcription_type/Results/"
# Local Functions



### Load and setup the data

In [2]:
# Load the setup transcripts data
# We have different species, so let's create a list to store the data
data_list = list()

# Search for the available files
files <- list.files(in_dir)

# Select files related to the species of interest
sel_species <- c("human")
file_human <- files[str_replace(files, "_.*", "") %in% sel_species]

# Load the data for each species
human_data<- read.table(file.path(in_dir, file_human), sep = "\t", header = TRUE, 
                                            stringsAsFactors = FALSE, quote = "", 
                                            comment.char = "", fill = TRUE, row.names = NULL)



head(human_data, 3)

,chromosome,ncRNA_id,ncrna_tss,ncrna_gene_name,ncrna_strand,gene_biotype,pcg_id,pcg_gene_name,pcg_tss,dna_distance,strand_distance,Family,is_TF
,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<lgl>
1,19,ENST00000221567,54532791,,1,lncRNA,ENST00000590333,BRSK1,55282072,749281,749281,NA,FALSE
2,19,ENST00000221567,54532791,,1,lncRNA,ENST00000635964,C19orf85,55464751,931960,931960,NA,FALSE
3,19,ENST00000221567,54532791,,1,lncRNA,ENST00000346968,CACNG6,53992878,-539913,-539913,NA,FALSE


Load the original ENSEMBL annotation to recover the pcg strand

In [3]:
# Load the raw-selected transcripts data
# We have different species, so let's create a list to store the data
data_list_raw = list()

# Search for the available files
files <- list.files(ensembl_raw_annotation_dir)

# Let's focus on human
files <- files[str_detect(files, "human")]

# Load the data for each species
for (file in files) {
    # Remove the underscore and everything after it to get the species names
    species_name <- str_replace(file, "_.*", "")
    data_list_raw[[species_name]] <- read.table(file.path(ensembl_raw_annotation_dir, file), sep = "\t", header = TRUE, 
                                            stringsAsFactors = FALSE, quote = "", 
                                            comment.char = "", fill = TRUE, row.names = NULL)
}

# We need to map start, end and strand information from the raw data to the selected data
sel_cols <- c('Transcript.stable.ID','Strand', 'Gene.stable.ID')
data_list_raw <- lapply(data_list_raw, function(df) df[, sel_cols])
head(data_list_raw[["human"]], 3)

,Transcript.stable.ID,Strand,Gene.stable.ID
,<chr>,<int>,<chr>
1,ENST00000456328,1,ENSG00000290825
2,ENST00000832823,1,ENSG00000290825
3,ENST00000832824,1,ENSG00000290825


Map the missing information to the human data

In [4]:
# Map the missing information
human_data <- human_data %>%
    # map the PCG information
    left_join(
        data_list_raw[["human"]],
        by = c("pcg_id" = "Transcript.stable.ID") 
    ) %>%
    dplyr::rename(
        pcg_strand = Strand,
        pcg_gene_id = Gene.stable.ID
    )
head(human_data, 3)

,chromosome,ncRNA_id,ncrna_tss,ncrna_gene_name,ncrna_strand,gene_biotype,pcg_id,pcg_gene_name,pcg_tss,dna_distance,strand_distance,Family,is_TF,pcg_strand,pcg_gene_id
,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<lgl>,<int>,<chr>
1,19,ENST00000221567,54532791,,1,lncRNA,ENST00000590333,BRSK1,55282072,749281,749281,NA,FALSE,1,ENSG00000160469
2,19,ENST00000221567,54532791,,1,lncRNA,ENST00000635964,C19orf85,55464751,931960,931960,NA,FALSE,-1,ENSG00000283567
3,19,ENST00000221567,54532791,,1,lncRNA,ENST00000346968,CACNG6,53992878,-539913,-539913,NA,FALSE,1,ENSG00000130433


In [5]:
rm(data_list_raw)
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1386535,74.1,4311572,230.3,4802801,256.5
Vcells,54582589,416.5,167258460,1276.1,167121189,1275.1


In [6]:
# Let's annotate the transcription direction sense
human_data <-human_data %>%
    mutate(
        transcription_type = case_when(
            ncrna_strand == pcg_strand ~ "Sense",
            ncrna_strand != pcg_strand ~ "Antisense",
            TRUE ~ "Unknown"
        )
    )
table(human_data$transcription_type)


Antisense     Sense 
  2239091   2111056 

Let's focus only in lncRNAs related to TF within the 10 Kb window

In [13]:
window_size <- 10000
# Select only observations related to lncRNA
data_human_selected <- human_data %>%
    filter(gene_biotype == "lncRNA") %>%
    filter(is_TF) %>%
    filter(abs(dna_distance) <= window_size)
table(data_human_selected$gene_biotype)
table(data_human_selected$transcription_type)


lncRNA 
 55454 


Antisense     Sense 
    41344     14110 

In [24]:
sense_type_data <- list()
for(sense_type in unique(data_human_selected$transcription_type)) {
    filtered_data <- data_human_selected %>%
        filter(transcription_type == sense_type) %>%
        select(pcg_gene_id, pcg_tss, chromosome) %>%
        distinct()
    cat("Number of unique PCG genes with", sense_type, ":\n",print(nrow(filtered_data)) )
    sense_type_data[[sense_type]] <- filtered_data
}

[1] 1629
Number of unique PCG genes with Antisense :
 1629[1] 569
Number of unique PCG genes with Sense :
 569

In [15]:
head(sense_type_data[["Sense"]], 3)

,pcg_gene_id
,<chr>
1,ENSG00000240445
2,ENSG00000284773
3,ENSG00000280778


In [ ]:
# Let's try to get unique genes... if possible
ovl_gene_ids <- intersect(sense_type_data[["Sense"]]$pcg_gene_id,
                          sense_type_data[["Antisense"]]$pcg_gene_id)
cat("Number of overlapping PCG genes between Sense and Antisense:", length(ovl_gene_ids), "\n")
# Let's remove overlapping genes from both sets
for(sense_type in names(sense_type_data)) {
    sense_type_data[[sense_type]] <- sense_type_data[[sense_type]] %>%
        filter(!pcg_gene_id %in% ovl_gene_ids) %>%
        filter(!duplicated(pcg_gene_id))
    cat("Number of unique PCG genes with", sense_type, "after removing overlaps:\n",print(nrow(sense_type_data[[sense_type]])) )
    update_date = "260119"
    # save the list of PCG genes
    write.table(sense_type_data[[sense_type]], file = paste0(out_dir, "human_unique_PCG_genes_", sense_type, "_", window_size, "bp_window_", update_date, ".tsv"),
                sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)
}

Number of overlapping PCG genes between Sense and Antisense: 0 
[1] 714
Number of unique PCG genes with Antisense after removing overlaps:
 714[1] 151
Number of unique PCG genes with Sense after removing overlaps:
 151

Let's save fixed regions around TSS for each transcription type for AME comparison

In [25]:
head(sense_type_data[["Sense"]],2)

,pcg_gene_id,pcg_tss,chromosome
,<chr>,<int>,<chr>
1,ENSG00000240445,18682262,17
2,ENSG00000284773,34985313,1


In [30]:
fixed_width = 1000
for(sense_type in names(sense_type_data)) {
    tmp_dat <- sense_type_data[[sense_type]] %>%
        # remove overlapping genes
        filter(!pcg_gene_id %in% ovl_gene_ids) %>%
        mutate(start = as.integer(floor(pcg_tss - fixed_width / 2)),
               end = as.integer(floor(pcg_tss + fixed_width / 2))) %>%
         select(chromosome, start, end) %>%
        mutate(chromosome = paste0("chr", chromosome)) %>%
        distinct()
    cat("Number of unique PCG genes regions with", sense_type, ":\n",print(nrow(tmp_dat)) )
    update_date = "260119"
    # save the list of PCG genes
    write.table(tmp_dat, file = paste0(out_dir, "human_fixed_width_TF_regions_", sense_type, "_", fixed_width, "bp_window_", update_date, ".bed"),
                    sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)
}

[1] 1628
Number of unique PCG genes regions with Antisense :
 1628[1] 569
Number of unique PCG genes regions with Sense :
 569